In [0]:
from pyspark.sql.functions import col
from pyspark.sql.functions import *

In [0]:
dbutils.widgets.text("catalog", "")
catalog = dbutils.widgets.get("catalog")

In [0]:
city_df=spark.read.table(f"{catalog}.silver_transport.city_silver")
trips_df=spark.read.table(f"{catalog}.silver_transport.trips_silver")


In [0]:
city_df.show()

In [0]:
trips_df.display()

In [0]:
print(city_df.columns)
print(trips_df.columns)

In [0]:
# Business Report 1: Total Sales by City
total_sales_by_city = (
    trips_df.select("city_id", "fare_amount")
    .join(city_df.select("city_id", "city_name", "state"), "city_id", "inner")
    .groupBy("city_name")
    .agg(sum("fare_amount").alias("total_sales"))
)
total_sales_by_city.createOrReplaceTempView("total_sales_by_city_view")
display(total_sales_by_city)

In [0]:
# Save total_sales_by_city_view to gold schema as a table
total_sales_by_city.write.mode("overwrite").saveAsTable(f"{catalog}.gold_transport.total_sales_by_city_view")

In [0]:
# Business Report 2: Average Passenger Rating by City
avg_passenger_rating_by_city = (
    trips_df.join(city_df, trips_df.city_id == city_df.city_id, "inner")
    .groupBy(city_df.city_name)
    .agg(round(avg("passenger_rating"), 2).alias("avg_passenger_rating"))
)
display(avg_passenger_rating_by_city)



In [0]:
# Save avg_passenger_rating_by_city to gold schema as a table
avg_passenger_rating_by_city.write.mode("overwrite").saveAsTable(f"{catalog}.gold_transport.avg_passenger_rating_by_city")

In [0]:
# Business Report 3: Trip Count by State and Passenger Type
trip_count_by_state_type = (
    trips_df.join(city_df, trips_df.city_id == city_df.city_id, "inner")
    .groupBy(city_df.state, trips_df.passenger_type)
    .count()
    .withColumnRenamed("count", "trip_count")
)
display(trip_count_by_state_type)

In [0]:
# Save avg_passenger_rating_by_city to gold schema as a table
trip_count_by_state_type.write.mode("overwrite").saveAsTable(f"{catalog}.gold_transport.trip_count_by_state_type")

In [0]:
# Business Report 4: Average Trip Distance by City
avg_distance_by_city = (
    trips_df.join(city_df, trips_df.city_id == city_df.city_id, "inner")
    .groupBy(city_df.city_name)
    .agg(round(avg("distance_travelled"), 2).alias("avg_distance_travelled"))
)
display(avg_distance_by_city)

In [0]:
avg_distance_by_city.write.mode("overwrite").saveAsTable(f"{catalog}.gold_transport.avg_distance_by_city")

In [0]:
# Business Report 5: Driver Rating Distribution by City
driver_rating_dist_by_city = (
    trips_df.select("city_id", "driver_rating")
    .join(city_df.select("city_id", "city_name"), "city_id", "inner")
    .groupBy("city_name", "driver_rating")
    .count()
    .withColumnRenamed("count", "rating_count")
)
display(driver_rating_dist_by_city)

In [0]:
driver_rating_dist_by_city.write.mode("overwrite").saveAsTable(f"{catalog}.gold_transport.driver_rating_dist_by_city")